# Databento Historical Data Cost and Acquisition Record

## Purpose

This notebook documents the data-choice process for the MES research project:
comparing schemas, estimating cost, acquiring a deliberately chosen immutable
raw source, and checking its coverage. It is not a routine pipeline notebook.

## Important safety decision

Cost-estimate cells are read-only. Download cells are intentionally guarded:
if their destination already exists they raise an error before making a paid
request or overwriting a raw DBN. To acquire a genuinely new source, choose a
new destination and document why it is a new version; do not reuse the
immutable production raw-file path.

## What successful output establishes

The historical record should show the selected schema, requested coverage,
stored raw-file location, record count, contract mapping, and known data-quality
conditions. Those facts are inputs to—not replacements for—the later processing
validation.

In [1]:
# Load the Databento API key from the private .env file in the project root.
# We only check whether the key was found—we never print the key itself.

import os
from dotenv import load_dotenv

load_dotenv("../.env")

api_key_found = bool(os.getenv("DATABENTO_API_KEY"))

print("Databento API key found:", api_key_found)

Databento API key found: True


In [2]:
# Create an authenticated Databento Historical API client.
# Databento will use the API key already loaded into the environment.

import databento as db

client = db.Historical()

In [4]:
# Define one complete MES trading session so the L0 and L1 cost estimates
# cover the exact same market period and are directly comparable.

dataset = "GLBX.MDP3"
symbol = "MES.v.0"
stype_in = "continuous"

# One complete MES trading session, expressed in UTC.
start = "2026-09-08T22:00:00Z"  # Tuesday 6:00 PM ET
end = "2026-09-09T21:00:00Z"    # Wednesday 5:00 PM ET

In [5]:
# Estimate the cost of one complete MES trading session using
# 1-minute OHLCV bars, which is our L0 comparison.

cost_l0 = client.metadata.get_cost(
    dataset=dataset,
    symbols=[symbol],
    schema="ohlcv-1m",
    stype_in=stype_in,
    start=start,
    end=end,
)

print(f"L0 OHLCV-1m estimated cost: ${cost_l0:.6f}")

L0 OHLCV-1m estimated cost: $0.005038


In [6]:
# Estimate the cost of the exact same MES trading session using MBP-1.
# MBP-1 contains trade events plus updates to the best bid and offer,
# giving us the Level 1 information needed for order-flow features.

cost_l1 = client.metadata.get_cost(
    dataset=dataset,
    symbols=[symbol],
    schema="mbp-1",
    stype_in=stype_in,
    start=start,
    end=end,
)

print(f"L1 MBP-1 estimated cost: ${cost_l1:.6f}")

L1 MBP-1 estimated cost: $1.144841


In [8]:
# Estimate the cost of roughly four months of historical MES Level 1 data.
# This uses MBP-1 and only requests a cost estimate—it does not download the data.

start_4m = "2026-05-11"
end_4m = "2026-09-11"

cost_l1_4m = client.metadata.get_cost(
    dataset=dataset,
    symbols=[symbol],
    schema="mbp-1",
    stype_in=stype_in,
    start=start_4m,
    end=end_4m,
)

print(f"Four-month L1 MBP-1 estimated cost: ${cost_l1_4m:.2f}")

Four-month L1 MBP-1 estimated cost: $124.49


In [9]:
# Estimate the cost of one year of MES trade-by-trade data.
# This is only a cost estimate and does not download or purchase the data.

start_trades_1y = "2025-09-11"
end_trades_1y = "2026-09-11"

cost_trades_1y = client.metadata.get_cost(
    dataset=dataset,
    symbols=[symbol],
    schema="trades",
    stype_in=stype_in,
    start=start_trades_1y,
    end=end_trades_1y,
)

print(f"One-year Trades estimated cost: ${cost_trades_1y:.2f}")

One-year Trades estimated cost: $129.78


In [10]:
# Estimate one year of MES TBBO data.
# TBBO gives us every trade together with the best bid/offer
# immediately before that trade, making it a strong candidate
# for matching our eventual IBKR Level 1 live-data pipeline.

cost_tbbo_1y = client.metadata.get_cost(
    dataset=dataset,
    symbols=[symbol],
    schema="tbbo",
    stype_in=stype_in,
    start=start_trades_1y,
    end=end_trades_1y,
)

print(f"One-year TBBO estimated cost: ${cost_tbbo_1y:.2f}")

One-year TBBO estimated cost: $216.30


## Guarded acquisition

The following examples are historical acquisition records, not cells to run
casually. Their destination checks intentionally stop existing-file runs before
the Databento request is made. This protects both the immutable raw source and
the project’s data credit.

In [11]:
# Estimate the cost of our one-session MES Trades test.
# This does NOT download data or use our historical-data credit.
# We are verifying the exact request before making our first real download.

cost_trades_test = client.metadata.get_cost(
    dataset="GLBX.MDP3",
    symbols=["MES.v.0"],
    schema="trades",
    stype_in="continuous",
    start="2026-09-08T22:00:00Z",
    end="2026-09-09T21:00:00Z",
)

print(f"One-session Trades test estimated cost: ${cost_trades_test:.6f}")

One-session Trades test estimated cost: $0.371683


In [12]:
# Download one complete MES session only when its destination is new.
# This explicit guard prevents an accidental paid repeat or raw-file overwrite.
from pathlib import Path

test_destination = Path('../data/mes_trades_2026-09-08_session.dbn')
if test_destination.exists():
    raise FileExistsError(
        f'{test_destination} already exists. This notebook will not re-download or overwrite it.'
    )

test_data = client.timeseries.get_range(
    dataset='GLBX.MDP3', symbols=['MES.v.0'], schema='trades',
    stype_in='continuous', start='2026-09-08T22:00:00Z', end='2026-09-09T21:00:00Z',
)
test_data.to_file(test_destination)
print(f'Test session downloaded: {test_destination}')

Test session downloaded successfully.


In [13]:
# Convert the downloaded Databento test session into a pandas DataFrame
# so we can inspect the structure of the raw MES trade records.
# This does not modify the saved .dbn file or make another API request.

trades_df = test_data.to_df()

print("Number of trade records:", len(trades_df))
print("\nColumns:")
print(trades_df.columns.tolist())

trades_df.head(10)

Number of trade records: 296943

Columns:
['ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side', 'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence', 'symbol']


In [14]:
# Perform a final sanity check on the one-session MES Trades sample.
# We verify the number of records, time coverage, price/size ranges,
# aggressor-side values, and whether any important fields are missing.

print("Trade records:", len(trades_df))
print("First timestamp:", trades_df.index.min())
print("Last timestamp:", trades_df.index.max())
print("Minimum price:", trades_df["price"].min())
print("Maximum price:", trades_df["price"].max())
print("Minimum trade size:", trades_df["size"].min())
print("Maximum trade size:", trades_df["size"].max())

print("\nAggressor-side counts:")
print(trades_df["side"].value_counts(dropna=False))

print("\nMissing values:")
print(trades_df[["price", "size", "side"]].isna().sum())

Trade records: 296943
First timestamp: 2026-09-08 22:00:00.012391321+00:00
Last timestamp: 2026-09-09 20:59:59.523624769+00:00
Minimum price: 7628.75
Maximum price: 7691.5
Minimum trade size: 1
Maximum trade size: 190

Aggressor-side counts:
side
B    148849
A    148093
N         1
Name: count, dtype: int64

Missing values:
price    0
size     0
side     0
dtype: int64


In [15]:
# Estimate the cost of our proposed full MES Trades research dataset.
# This does NOT download any data or consume our historical-data credit.
# We are checking whether this start date gives us the amount of history
# we want while leaving some credit available for future experiments.

full_start = "2025-10-20"
full_end = "2026-09-11"

cost_trades_full = client.metadata.get_cost(
    dataset="GLBX.MDP3",
    symbols=["MES.v.0"],
    schema="trades",
    stype_in="continuous",
    start=full_start,
    end=full_end,
)

print("Proposed date range:", full_start, "to", full_end)
print(f"Estimated full Trades cost: ${cost_trades_full:.2f}")

Proposed date range: 2025-10-20 to 2026-09-11
Estimated full Trades cost: $118.99


In [16]:
# Estimate a slightly larger MES Trades dataset while staying just under
# the remaining Databento credit balance.

full_start = "2025-10-08"
full_end = "2026-09-11"

cost_trades_full = client.metadata.get_cost(
    dataset="GLBX.MDP3",
    symbols=["MES.v.0"],
    schema="trades",
    stype_in="continuous",
    start=full_start,
    end=full_end,
)

print("Proposed date range:", full_start, "to", full_end)
print(f"Estimated full Trades cost: ${cost_trades_full:.2f}")

Proposed date range: 2025-10-08 to 2026-09-11
Estimated full Trades cost: $123.95


In [17]:
# Estimate the cost of MES Trades data from October 7, 2025 through
# the most recent completed data in our research period.
# This is only a cost estimate and does NOT download any data.

full_start = "2025-10-07"
full_end = "2026-09-11"

cost_trades_full = client.metadata.get_cost(
    dataset="GLBX.MDP3",
    symbols=["MES.v.0"],
    schema="trades",
    stype_in="continuous",
    start=full_start,
    end=full_end,
)

print("Proposed date range:", full_start, "to", full_end)
print(f"Estimated full Trades cost: ${cost_trades_full:.2f}")

Proposed date range: 2025-10-07 to 2026-09-11
Estimated full Trades cost: $124.32


In [18]:
# Download the full raw history only to an explicitly new, empty destination.
# The established production DBN is immutable. This guard runs before any paid
# request, so re-running the notebook cannot silently spend credit or overwrite it.
from pathlib import Path

raw_destination = Path('../data/mes_trades_2025-10-07_to_2026-09-11.dbn')
if raw_destination.exists():
    raise FileExistsError(
        f'{raw_destination} already exists and is immutable. Choose and document a new destination '
        'before intentionally requesting different raw data.'
    )

client.timeseries.get_range(
    dataset='GLBX.MDP3', symbols=['MES.v.0'], schema='trades',
    stype_in='continuous', start='2025-10-07', end='2026-09-11', path=raw_destination,
)
print(f'Full MES Trades dataset downloaded: {raw_destination}')

/var/folders/cw/0s2xtqxs2tj9_k3253g7vbpw0000gn/T/ipykernel_13088/1490362878.py:7: BentoWarning: The streaming request contained one or more days which have reduced quality: 2025-11-28 (degraded), 2026-01-31 (degraded), 2026-03-15 (degraded)... See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  client.timeseries.get_range(


Full MES Trades dataset downloaded successfully.


In [19]:
# Open the saved raw DBN file from disk without loading the full dataset
# into a pandas DataFrame. We will use this to verify the file structure
# and inspect a small sample safely.

import databento as db

full_data = db.DBNStore.from_file(
    "../data/mes_trades_2025-10-07_to_2026-09-11.dbn"
)

print(full_data)

<DBNStore(schema=trades)>


In [20]:
# Stream through the full DBN file one record at a time.
# This verifies the total number of trade records and the actual time
# coverage without loading the entire 1.7 GB dataset into memory.

record_count = 0
first_timestamp = None
last_timestamp = None

for record in full_data:
    # Save the timestamp from the first trade we encounter.
    if first_timestamp is None:
        first_timestamp = record.ts_event

    # Because the DBN records are chronological, this will eventually
    # contain the timestamp of the final trade in the dataset.
    last_timestamp = record.ts_event
    record_count += 1

print(f"Total trade records: {record_count:,}")
print("First event timestamp:", first_timestamp)
print("Last event timestamp:", last_timestamp)

Total trade records: 99,319,450
First event timestamp: 1759795200043782871
Last event timestamp: 1789084799790465427


In [21]:
# Convert the raw nanosecond timestamps into readable UTC dates and times.
# Databento stores ts_event as nanoseconds since the Unix epoch, so pandas
# can translate these integers without loading the full trade dataset again.

import pandas as pd

first_datetime = pd.to_datetime(first_timestamp, unit="ns", utc=True)
last_datetime = pd.to_datetime(last_timestamp, unit="ns", utc=True)

print(f"Total trade records: {record_count:,}")
print("First trade:", first_datetime)
print("Last trade:", last_datetime)

Total trade records: 99,319,450
First trade: 2025-10-07 00:00:00.043782871+00:00
Last trade: 2026-09-10 23:59:59.790465427+00:00


In [22]:
# Inspect the symbology mapping stored inside the DBN file.
# MES.v.0 is a continuous contract, so Databento maps it to different
# actual MES futures contracts over time as the lead contract changes.
#
# This reads the small metadata mapping only; it does NOT scan all
# 99 million trade records again.

print(full_data.symbology)

{'symbols': ['MES.v.0'], 'stype_in': 'continuous', 'stype_out': 'instrument_id', 'start_date': '2025-10-07', 'end_date': '2026-09-11', 'partial': [], 'not_found': [], 'mappings': {'MES.v.0': [{'start_date': datetime.date(2025, 10, 7), 'end_date': datetime.date(2025, 12, 17), 'symbol': '42004164'}, {'start_date': datetime.date(2025, 12, 17), 'end_date': datetime.date(2026, 3, 18), 'symbol': '42003800'}, {'start_date': datetime.date(2026, 3, 18), 'end_date': datetime.date(2026, 6, 17), 'symbol': '42005163'}, {'start_date': datetime.date(2026, 6, 17), 'end_date': datetime.date(2026, 9, 11), 'symbol': '42003239'}]}}


In [23]:
# Translate the four Databento instrument IDs used by MES.v.0
# into the actual CME futures contract symbols.
#
# This is a symbology lookup only. It does NOT download trade data
# and does not scan our 99 million stored trade records.

contract_ids = [
    "42004164",
    "42003800",
    "42005163",
    "42003239",
]

contract_symbols = client.symbology.resolve(
    dataset="GLBX.MDP3",
    symbols=contract_ids,
    stype_in="instrument_id",
    stype_out="raw_symbol",
    start_date="2025-10-07",
    end_date="2026-09-11",
)

print(contract_symbols)

{'result': {'42004164': [{'d0': '2025-10-07', 'd1': '2026-01-04', 's': 'MESZ5'}, {'d0': '2026-01-04', 'd1': '2026-06-01', 's': 'ZYECK6'}, {'d0': '2026-06-01', 'd1': '2026-08-16', 's': '1SN608'}, {'d0': '2026-08-16', 'd1': '2026-09-11', 's': 'NDKX0'}], '42003800': [{'d0': '2025-10-07', 'd1': '2026-09-10', 's': 'MESH6'}], '42005163': [{'d0': '2025-10-07', 'd1': '2026-07-02', 's': 'MESM6'}, {'d0': '2026-07-02', 'd1': '2026-08-30', 's': 'NWDU609'}, {'d0': '2026-08-30', 'd1': '2026-09-11', 's': 'ESXZ6'}], '42003239': [{'d0': '2025-10-07', 'd1': '2026-09-11', 's': 'MESU6'}]}, 'symbols': ['42004164', '42003800', '42005163', '42003239'], 'stype_in': 'instrument_id', 'stype_out': 'raw_symbol', 'start_date': '2025-10-07', 'end_date': '2026-09-11', 'partial': [], 'not_found': [], 'message': 'OK', 'status': 0}


In [25]:
# Check Databento's official data-quality status for every date covered
# by our historical MES dataset.
#
# We print only dates that are NOT marked "available" so we can identify
# degraded, missing, or pending dates that may need special treatment.
# This is a metadata request and does NOT scan our 99 million trade records.

dataset_conditions = client.metadata.get_dataset_condition(
    dataset="GLBX.MDP3",
    start_date="2025-10-07",
    end_date="2026-09-10",
)

problem_dates = [
    day for day in dataset_conditions
    if day["condition"] != "available"
]

print(f"Total dates checked: {len(dataset_conditions)}")
print(f"Dates not fully available: {len(problem_dates)}")

for day in problem_dates:
    print(
        day["date"],
        "-",
        day["condition"],
        "- last modified:",
        day["last_modified_date"],
    )

Total dates checked: 331
Dates not fully available: 9
2025-11-28 - degraded - last modified: 2026-06-09
2026-01-31 - degraded - last modified: 2026-06-04
2026-03-15 - degraded - last modified: 2026-06-05
2026-03-16 - degraded - last modified: 2026-06-06
2026-03-21 - degraded - last modified: 2026-05-18
2026-04-10 - degraded - last modified: 2026-05-21
2026-05-24 - degraded - last modified: 2026-06-05
2026-07-30 - degraded - last modified: 2026-07-31
2026-08-29 - degraded - last modified: 2026-08-30
